In [2]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

THU_MUC_GOC = Path("..").resolve()
THU_MUC_DAC_TRUNG = THU_MUC_GOC / "data" / "features"
THU_MUC_MODEL = THU_MUC_GOC / "model"
THU_MUC_REPORT = THU_MUC_GOC / "report"

THU_MUC_MODEL.mkdir(parents=True, exist_ok=True)
THU_MUC_REPORT.mkdir(parents=True, exist_ok=True)

X_train = np.load(THU_MUC_DAC_TRUNG / "X_train.npy")
y_train = np.load(THU_MUC_DAC_TRUNG / "y_train.npy")
print(f"[Random Forest] Đã nạp X_train: {X_train.shape} | y_train: {y_train.shape}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

bat_dau = time.time()

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

param_rf = {
    "n_estimators": [150, 300],
    "max_depth": [None, 25],
    "min_samples_split": [2, 5],
}

grid_rf = GridSearchCV(
    rf,
    param_grid=param_rf,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)
grid_rf.fit(X_train, y_train)

thoi_gian = time.time() - bat_dau
joblib.dump(grid_rf.best_estimator_, THU_MUC_MODEL / "rf_model.pkl")

ket_qua = {
    "model": "RandomForest",
    "best_params": grid_rf.best_params_,
    "cv_f1_macro": round(float(grid_rf.best_score_), 4),
    "train_time_sec": round(thoi_gian, 1),
}

with open(THU_MUC_REPORT / "cv_rf.json", "w", encoding="utf-8") as f:
    json.dump(ket_qua, f, ensure_ascii=False, indent=2)

print("\nHoàn tất Random Forest:", ket_qua)

[Random Forest] Đã nạp X_train: (12888, 1280) | y_train: (12888,)
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Hoàn tất Random Forest: {'model': 'RandomForest', 'best_params': {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 300}, 'cv_f1_macro': 0.9396, 'train_time_sec': 210.1}
